In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Plot styling
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
df = pd.read_csv('../data/raw/raw_analyst_ratings.csv')

print("Shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

In [ ]:


# Parse dates properly
df['date'] = pd.to_datetime(df['date'], format='mixed', utc=True)

# Basic info
print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nData types:\n", df.dtypes)
print("\nDate range:", df['date'].min(), "to", df['date'].max())

In [ ]:
# Headline length analysis
df['headline_length'] = df['headline'].str.len()
df['word_count'] = df['headline'].str.split().str.len()

print("Headline Character Length Stats:")
print(df['headline_length'].describe())

print("\nHeadline Word Count Stats:")
print(df['word_count'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['headline_length'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Headline Character Length Distribution')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['word_count'], bins=40, color='coral', edgecolor='white')
axes[1].set_title('Headline Word Count Distribution')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../reports/figures/headline_length_distribution.png', dpi=150)
plt.show()

### 📊 Headline Length Analysis

#### Key Findings

**Character Length:**
- The average headline is **73 characters** long (std: 40), with a median of **64 characters**
- 50% of headlines fall between **47 and 87 characters** — consistent with typical news headline conventions
- The distribution is **right-skewed**, with a long tail extending to a maximum of **512 characters**
- The minimum of **3 characters** suggests some data quality issues worth investigating

**Word Count:**
- Headlines average **11 words** (std: 6), with a median of **10 words**
- 50% of headlines contain between **7 and 13 words**
- The distribution follows a **log-normal shape**, peaking around 10 words
- Very few headlines exceed 30 words, with a maximum of 77 words

#### Insight
The tight concentration of headline lengths (7–13 words, 47–87 characters) reflects the standardized, concise nature of financial news writing. The right skew and long tail suggest a minority of articles contain unusually detailed or compound headlines — these may represent earnings reports, regulatory filings, or multi-stock analyses, whic

## Publisher Analysis

#### Frequency & Distribution Analysis:

In [ ]:
# ── PUBLISHER FREQUENCY ANALYSIS ──────────────────────────────────────────────

# Unique publisher count
print(f"Total unique publishers: {df['publisher'].nunique()}")

# Top 20 publishers by article count
publisher_counts = df['publisher'].value_counts()
top20 = publisher_counts.head(20)
print("\nTop 20 Publishers:\n", top20)

# Plot
plt.figure(figsize=(14, 7))
top20.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top 20 Most Active Publishers', fontsize=15)
plt.xlabel('Number of Articles')
plt.ylabel('Publisher')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/figures/top_publishers.png', dpi=150)
plt.show()

####  Email domain extraction

In [ ]:
# ── EMAIL DOMAIN EXTRACTION ────────────────────────────────────────────────────

# Normalize publisher names
df['publisher_clean'] = df['publisher'].str.lower().str.strip()

# Extract domain from email-based publisher names
def extract_publisher(name):
    if '@' in str(name):
        return name.split('@')[1].strip()
    return name

df['publisher_clean'] = df['publisher_clean'].apply(extract_publisher)

# Summary
email_mask = df['publisher'].str.contains('@', na=False)
print(f"Articles with email-based publisher names: {email_mask.sum()}")
print(f"Unique domains extracted: {df[email_mask]['publisher_clean'].nunique()}")
print("\nTop 10 email domains:\n", df[email_mask]['publisher_clean'].value_counts().head(10))

#### Publisher-Stock Heatmap + Publishing Cadence:

In [ ]:
# ── PUBLISHER-STOCK COVERAGE & CADENCE ────────────────────────────────────────

# Filter to our 7 yfinance stocks
yfinance_stocks = ['AAPL', 'AMZN', 'GOOG', 'META', 'MSFT', 'NVDA', 'TSLA']
df_focus = df[df['stock'].isin(yfinance_stocks)]

# Publisher-stock heatmap (top 15 publishers)
top15_pub = df_focus['publisher_clean'].value_counts().head(15).index
heatmap_data = df_focus[df_focus['publisher_clean'].isin(top15_pub)]\
    .groupby(['publisher_clean', 'stock']).size().unstack(fill_value=0)

plt.figure(figsize=(14, 8))
sns.heatmap(heatmap_data, cmap='YlOrRd', annot=True, fmt='d', linewidths=0.5)
plt.title('Top 15 Publishers vs Stock Coverage', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/publisher_stock_heatmap.png', dpi=150)
plt.show()

# Publishing cadence
df['hour'] = df['date'].dt.hour
df['is_market_hours'] = df['hour'].between(13, 20)  # 9am-4pm EST = 13-20 UTC

top10_pub = publisher_counts.head(10).index
cadence = df[df['publisher'].isin(top10_pub)]\
    .groupby('publisher_clean')['is_market_hours'].mean() * 100

print("% of articles published during market hours (top 10 publishers):")
print(cadence.sort_values(ascending=False).round(2))

## 📰 Publisher Analysis — Key Findings

- **1,034 unique publishers**, heavily concentrated — top 5 dominate the majority of articles
- **Paul Quintaro (228K)** and **Lisa Levin (187K)** alone represent a significant portion of the dataset, introducing potential **sentiment bias**
- `Benzinga Newsdesk` vs `Benzinga_Newsdesk` is a **data quality flag** — needs merging before modeling
- **8,088 email-based publishers** resolve to just 8 domains, almost entirely `benzinga.com` — confirming this is largely a Benzinga-sourced dataset
- **TSLA and NVDA** are the most covered stocks — correlation analysis will be most reliable for these two
- Most news is published **outside market hours** — sentiment likely influences **next day's price** rather than same-day close

> ⚠️ **Preprocessing Flag:** `Benzinga_Newsdesk` and `Benzinga Newsdesk` are the same publisher
> split by an underscore. This will be normalized in the preprocessing stage before modeling.

#### Date Feature Extraction

In [ ]:
# ── DATE FEATURE EXTRACTION ────────────────────────────────────────────────────

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.day_name()
df['hour'] = df['date'].dt.hour

print("Articles per year:")
print(df['year'].value_counts().sort_index())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# 1. Yearly trend
yearly = df.groupby('year').size()
axes[0].bar(yearly.index, yearly.values, color='steelblue', edgecolor='white')
axes[0].set_title('Number of Articles Per Year', fontsize=14)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Article Count')

# 2. Day of week distribution
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow = df['day_of_week'].value_counts().reindex(day_order)
axes[1].bar(dow.index, dow.values, color='mediumseagreen', edgecolor='white')
axes[1].set_title('Article Count by Day of Week', fontsize=14)
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Article Count')

plt.tight_layout()
plt.savefig('../reports/figures/publication_trends.png', dpi=150)
plt.show()

In [ ]:
df['month_num'] = df['date'].dt.month
monthly_heatmap = df.groupby(['year', 'month_num']).size().unstack(fill_value=0)

plt.figure(figsize=(14, 7))
sns.heatmap(monthly_heatmap, cmap='YlOrRd', annot=True, fmt='d',
            linewidths=0.5,
            xticklabels=['Jan','Feb','Mar','Apr','May','Jun',
                         'Jul','Aug','Sep','Oct','Nov','Dec'])
plt.title('Monthly Article Volume Heatmap (Year vs Month)', fontsize=14)
plt.xlabel('Month')
plt.ylabel('Year')
plt.tight_layout()
plt.savefig('../reports/figures/monthly_heatmap.png', dpi=150)
plt.show()

In [ ]:
import plotly.graph_objects as go

# ── SPIKE DETECTION ────────────────────────────────────────────────────────────

# 1. Prepare Daily Counts
daily = df.groupby(df['date'].dt.date).size().reset_index(name='article_count')
daily['date'] = pd.to_datetime(daily['date'])
daily = daily.sort_values('date').reset_index(drop=True)

# 2. Rolling Statistics
daily['rolling_mean'] = daily['article_count'].rolling(window=30, min_periods=1).mean()
daily['rolling_std']  = daily['article_count'].rolling(window=30, min_periods=1).std().fillna(0)
daily['upper_band']   = daily['rolling_mean'] + 3 * daily['rolling_std']

# 3. Classify Spikes
daily['is_spike'] = daily['article_count'] > daily['upper_band']

# 4. Hover Text
daily['hover'] = daily.apply(
    lambda r: f"Date: {r['date'].strftime('%Y-%m-%d')}<br>"
              f"Articles: {r['article_count']:,}<br>"
              f"{'🔴 SPIKE DETECTED' if r['is_spike'] else ''}",
    axis=1
)

spikes = daily[daily['is_spike']]
top_3  = daily.nlargest(3, 'article_count')

# 5. Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=daily['date'], y=daily['article_count'],
    fill='tozeroy', fillcolor='rgba(70,130,180,0.15)',
    line=dict(color='steelblue', width=1),
    name='Daily Volume', hovertext=daily['hover'],
    hoverinfo='text'
))

fig.add_trace(go.Scatter(
    x=daily['date'], y=daily['rolling_mean'],
    line=dict(color='orange', width=2, dash='dash'),
    name='30-Day Rolling Mean'
))

fig.add_trace(go.Scatter(
    x=daily['date'], y=daily['upper_band'],
    line=dict(color='red', width=1.5, dash='dot'),
    name='Spike Threshold (Mean + 3σ)'
))

fig.add_trace(go.Scatter(
    x=spikes['date'], y=spikes['article_count'],
    mode='markers',
    marker=dict(color='red', size=6, symbol='circle'),
    name=f'Spikes ({len(spikes)})',
    hovertext=spikes['hover'], hoverinfo='text'
))

for _, row in top_3.iterrows():
    fig.add_annotation(
        x=row['date'], y=row['article_count'],
        text=row['date'].strftime('%Y-%m-%d'),
        showarrow=True, arrowhead=2,
        arrowcolor='darkred',
        font=dict(size=10, color='darkred'),
        ay=-40
    )

fig.update_layout(
    title='Financial News Volume: Spike & Anomaly Detection (2009–2020)',
    xaxis_title='Date',
    yaxis_title='Daily Article Count',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    template='plotly_white',
    height=500
)

fig.write_html('../reports/figures/spike_detection.html')
fig.show()

# 6. Summary
print(f"Total spikes detected: {len(spikes)}")
print(f"\nTop 10 busiest days:")
print(daily.nlargest(10, 'article_count')[['date', 'article_count', 'is_spike']].to_string(index=False))

## 📅 Publication Date Analysis — Temporal Trends

### Articles Per Year
The dataset spans **2009 to mid-2020** with a clear three-phase lifecycle:
- **Ramp-up (2009–2011):** Volume grew rapidly from ~11K to ~131K articles annually as the data pipeline matured
- **Stable Era (2012–2019):** Consistent output of 120K–150K articles per year — highly reliable for long-term analysis
- **2020 truncation:** Only ~105K articles due to the dataset ending in June 2020

### Day of Week Pattern
News is overwhelmingly a **weekday phenomenon** — Tuesday, Wednesday and Thursday peak above 300K total articles while Saturday and Sunday are near zero. This confirms the dataset follows institutional market cycles and means our correlation analysis will be strongest Monday–Friday, with potential **Monday lag effects** from weekend news.

### Monthly Heatmap
Three patterns stand out:
- **Data sparse before August 2009** — treat pre-2010 data with caution
- **2018–2019 golden era** — most consistent high-volume period, ideal for reliable sentiment modeling
- **March 2020 outlier (24,995 articles)** — the single hottest cell in the entire dataset, directly corresponding to the COVID-19 market crash onset

### Spike Detection
Only **6 statistical spikes** (Mean + 3σ) were detected across 11 years:
- **Early spikes (2009–2010):** Likely data pipeline artifacts during ramp-up, not real market events
- **March 12, 2020 — the Black Swan:** Daily volume hit **2,739 articles (~6x the historical average)**, pinpointing the exact day global COVID-19 panic peaked in financial markets

> 💡 **Key Implication for Correlation Analysis:** The March 2020 cluster is the highest-signal window in the entire dataset. If news sentiment drove stock prices at any point in this 11-year period, it happened here. This will be our primary validation window when measuring sentiment-to-price correlation.